# Notebook 12 — Aggregation and reconciliation

**ClinicalShield v2 · Revision pipeline · CPU · minutes**

## What this exists to prevent

v1's tables were transcribed by hand from notebook output, and drifted from the figures they
described. Reviewers 1 and 4 found four such inconsistencies between them:

- Table 2 reported 6,627 samples; Figure 3 summed to 6,453
- Table 6 reported 92.23% accuracy and 1.01% FPR against a confusion matrix implying 86.98%
  and 10.83%
- baseline metrics conflicted with the repository's own JSON, and were in places
  arithmetically impossible given the stated supports
- CEPS values appeared with two different sample counts

None of those is a modelling error. All four are transcription errors, and all four were
avoidable.

## The fix

Every notebook writes a result file. This notebook reads those files, cross-checks each
number against the same quantity derived independently elsewhere, and emits
`all_paper_numbers.json`.

**The manuscript is written from that file.** A number without a ledger entry does not go in
the paper.

## The reconciliation gate

Cell 4 compares numbers across notebooks: the corpus manifest against the attack statistics,
the attack statistics against the partition sizes, the partition sizes against the pooled
confusion matrix, and every reported metric against the matrix it came from.

**A mismatch raises.** The v1 failure cannot survive a run of this notebook.

## The ledger

Each entry carries its value, its standard deviation where it has one, the notebook that
produced it, the partition it was measured on, its n, and the tables and sections it appears
in.

That last field matters for a specific v1 failure: Table 6 paired a plaintext-partition FPR
with a full-pipeline recall in one row. With partition recorded per number, that row cannot
be assembled.

## Reviewer coverage

Cell 6 maps all 31 reviewer points to the notebook addressing each and the ledger entries
evidencing it. That is the skeleton of the point-by-point response, and it shows immediately
whether any point still lacks data.

## Runtime
CPU. Minutes. Reads JSON only.

In [ ]:
# ── Cell 1 · Configuration ────────────────────────────────────────────────────
CONFIG = {
    "SEED": 42,

    # Every result file produced by the pipeline. NB12 reads only these: no
    # number in the manuscript is retyped from a notebook's printed output.
    "SOURCES": {
        "corpus":      "data/corpus/manifest.json",
        "corpus_stats":"data/stats/corpus_stats.json",
        "attacks":     "data/stats/attack_stats.json",
        "split":       "data/stats/split_stats.json",
        "module1":     "data/results/module1_results.json",
        "module2":     "data/results/module2_results.json",
        "module3":     "data/results/module3_results.json",
        "module4":     "data/results/module4_results.json",
        "baselines":   "data/results/baseline_results.json",
        "end_to_end":  "data/results/end_to_end_results.json",
        "mpib":        "data/results/mpib_results.json",
        "sweeps":      "data/results/sweep_results.json",
    },

    "USE_DRIVE":      True,
    "DRIVE_DIR":      "/content/drive/MyDrive/ClinicalShield_v2",
    "PUSH_TO_GITHUB": True,
    "GITHUB_REPO":    "NehlTech/ClinicalShield",
    "GITHUB_BRANCH":  "v2-revision",
    "GIT_USER_NAME":  "Adu-Boahene Bright",
    "GIT_USER_EMAIL": "baduboahene@st.knust.edu.gh",
}
SEED = CONFIG["SEED"]
print("sources to reconcile: %d" % len(CONFIG["SOURCES"]))
for k, v in CONFIG["SOURCES"].items():
    print("  %-14s %s" % (k, v))


In [ ]:
# ── Cell 2 · Environment and inputs ───────────────────────────────────────────
# NB11 retrains PubMedBERT across sweep arms, so it needs the dataset and
# Module 1 only. No checkpoints are loaded; each arm trains from scratch.
import sys, os, json, random, hashlib, subprocess, shutil, time, math
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

random.seed(SEED); np.random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT, REPO_DIR = None, None

if IN_COLAB:
    if CONFIG["USE_DRIVE"]:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(CONFIG["DRIVE_DIR"]); DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print("drive :", DRIVE_ROOT)
    repo_name = CONFIG["GITHUB_REPO"].split("/")[-1]
    REPO_DIR = Path("/content") / repo_name
    if not REPO_DIR.exists():
        try:
            from google.colab import userdata
            tok = userdata.get("GITHUB_TOKEN")
            r = subprocess.run(["git","clone","-q",
                "https://" + tok + "@github.com/" + CONFIG["GITHUB_REPO"] + ".git",
                str(REPO_DIR)], capture_output=True, text=True)
            print("clone :", "ok" if r.returncode == 0 else r.stderr[:200])
        except Exception as e:
            print("clone skipped:", type(e).__name__)
    else:
        print("clone : already present")
    if REPO_DIR.exists():
        for k, v in [("user.name", CONFIG["GIT_USER_NAME"]),
                     ("user.email", CONFIG["GIT_USER_EMAIL"])]:
            subprocess.run(["git","-C",str(REPO_DIR),"config",k,v], check=False)
        subprocess.run(["git","-C",str(REPO_DIR),"checkout","-q",
                        CONFIG["GITHUB_BRANCH"]], check=False, capture_output=True)
        subprocess.run(["git","-C",str(REPO_DIR),"pull","-q","origin",
                        CONFIG["GITHUB_BRANCH"]], check=False, capture_output=True)
    ROOT = REPO_DIR if REPO_DIR.exists() else Path("/content")
else:
    ROOT = Path.cwd()
    while not (ROOT/".git").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT/".git").exists():
        ROOT = Path.cwd()

DIRS = {"dataset": ROOT/"data"/"dataset", "stats": ROOT/"data"/"stats",
        "results": ROOT/"data"/"results", "src": ROOT/"src"/"clinicalshield",
        "ckpt": ROOT/"checkpoints"}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
print("root  :", ROOT)

import re
print("NB12 reads JSON only: no model, no dataset, no GPU required")
import re


## Loading

A missing source is reported rather than silently skipped, because a table written from an
absent file is exactly how the v1 inconsistencies arose.

In [ ]:
# ── Cell 3 · Load every result file ───────────────────────────────────────────
# A missing source is reported, not silently skipped. v1's numbers drifted
# because tables were typed by hand from notebook output; here the manuscript
# reads one file, and that file is built only from these.

S = {}
missing = []
for name, rel in CONFIG["SOURCES"].items():
    p = ROOT / rel
    if not p.exists() and DRIVE_ROOT and (DRIVE_ROOT / rel).exists():
        p.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(DRIVE_ROOT / rel, p)
    if p.exists():
        S[name] = json.load(open(p))
        print("  %-14s loaded  (%.0f KB)" % (name, p.stat().st_size/1024))
    else:
        missing.append(name)
        print("  %-14s MISSING at %s" % (name, rel))

if missing:
    print("")
    print("missing sources: %s" % missing)
    print("Any table depending on these cannot be written. Run the notebook that")
    print("produces each before continuing.")


## Reconciliation

The gate. Each check derives a number two ways and compares them.

Tolerances are tight — 1e-6 for counts, 1e-4 for rates — because these quantities should
agree exactly, not approximately. Every seed is evaluated on the same partition, so the mean
of per-seed rates and the rate from pooled counts are mathematically identical, not merely
close.

In [ ]:
# ── Cell 4 · Cross-notebook reconciliation ────────────────────────────────────
# The failure this exists to prevent: v1 reported 6,627 samples in Table 2 while
# Figure 3 summed to 6,453, and 92.23% accuracy against a confusion matrix
# implying 86.98%. Reviewers 1 and 4 both found it.
#
# Every check below compares a number from one notebook against the same number
# derived independently elsewhere. A mismatch raises.

checks = []

def check(name, a, b, tol=1e-6, note=""):
    ok = (a is None and b is None) or (
        a is not None and b is not None and abs(float(a) - float(b)) <= tol)
    checks.append({"check": name, "value_a": a, "value_b": b,
                   "tolerance": tol, "passed": bool(ok), "note": note})
    print("  %-46s %-14s %-14s %s"
          % (name, "%.6g" % a if a is not None else "-",
             "%.6g" % b if b is not None else "-", "OK" if ok else "MISMATCH"))
    return ok

print("  %-46s %-14s %-14s %s" % ("check", "source A", "source B", ""))
print("  " + "-"*92)

# corpus -> attacks
if "corpus" in S and "attacks" in S:
    check("rag documents: manifest vs attack stats",
          S["corpus"]["counts"]["rag_documents"],
          S["attacks"]["counts"]["source_documents"])

# attacks -> split
if "attacks" in S and "split" in S:
    tot_split = sum(S["split"]["partitions"][k]["chunks"] for k in ["train","val","test"])
    check("total chunks: attack stats vs split partitions",
          S["attacks"]["counts"]["chunks"], tot_split)
    adv_split = sum(S["split"]["partitions"][k]["adversarial"]
                    for k in ["train","val","test"])
    check("adversarial: attack stats vs split partitions",
          S["attacks"]["counts"]["adversarial"], adv_split)
    grp_split = sum(S["split"]["partitions"][k]["groups"] for k in ["train","val","test"])
    check("groups: attack stats vs split partitions",
          S["attacks"]["counts"]["groups"], grp_split)

# split -> module2 (test partition size must match the confusion matrix)
if "split" in S and "module2" in S:
    n_test = S["split"]["partitions"]["test"]["chunks"]
    pc = S["module2"]["pooled_confusion"]
    n_seeds = len(S["module2"]["seeds"])
    check("test chunks x seeds vs pooled confusion total",
          n_test * n_seeds, sum(pc.values()))
    adv_test = S["split"]["partitions"]["test"]["adversarial"]
    check("test adversarial x seeds vs pooled (tp+fn)",
          adv_test * n_seeds, pc["tp"] + pc["fn"])
    ben_test = S["split"]["partitions"]["test"]["benign"]
    check("test benign x seeds vs pooled (tn+fp)",
          ben_test * n_seeds, pc["tn"] + pc["fp"])

# module2 internal arithmetic -- the v1 failure, re-asserted here
if "module2" in S:
    pc = S["module2"]["pooled_confusion"]
    tot = sum(pc.values())
    check("accuracy: reported vs confusion matrix",
          S["module2"]["summary"]["accuracy"]["mean"],
          (pc["tn"] + pc["tp"]) / tot, tol=1e-4,
          note="v1 reported 92.23% against a matrix implying 86.98%")
    check("recall: reported vs confusion matrix",
          S["module2"]["summary"]["recall"]["mean"],
          pc["tp"] / (pc["tp"] + pc["fn"]), tol=1e-4)
    check("fpr: reported vs confusion matrix",
          S["module2"]["summary"]["fpr"]["mean"],
          pc["fp"] / (pc["fp"] + pc["tn"]), tol=1e-4)
    check("precision: reported vs confusion matrix",
          S["module2"]["summary"]["precision"]["mean"],
          pc["tp"] / (pc["tp"] + pc["fp"]), tol=1e-4)

# baselines: every row's own arithmetic
if "baselines" in S:
    bad = []
    for name, r in S["baselines"]["baselines"].items():
        tot = r["tn"] + r["fp"] + r["fn"] + r["tp"]
        if abs(r["accuracy"] - (r["tn"]+r["tp"])/tot) > 1e-6:
            bad.append(name + ":accuracy")
        den = r["tp"] + r["fn"]
        if den and abs(r["recall"] - r["tp"]/den) > 1e-6:
            bad.append(name + ":recall")
    check("baseline rows consistent with their own matrices",
          0, len(bad), note="v1 baselines conflicted with the repository JSON (R4-4)")
    if bad:
        print("      inconsistent: %s" % bad)

# module1 probe pools must not overlap
if "module1" in S and "pool_separation" in S["module1"]:
    check("module1 dev/test probe pools share no groups",
          0, S["module1"]["pool_separation"]["shared_groups"])

# leakage gates
if "split" in S:
    lk = S["split"]["leakage_checks"]
    check("groups spanning partitions", 0, lk["groups_spanning_partitions"])
    check("documents spanning partitions", 0, lk["documents_spanning_partitions"])
    check("templates spanning partitions", 0,
          lk.get("templates_spanning_partitions", 0))

n_pass = sum(c["passed"] for c in checks)
print("")
print("  %d/%d checks passed" % (n_pass, len(checks)))
failed = [c for c in checks if not c["passed"]]
if failed:
    print("")
    print("  FAILED:")
    for c in failed:
        print("    %s: %s vs %s" % (c["check"], c["value_a"], c["value_b"]))
    raise AssertionError("%d reconciliation checks failed -- do not write the "
                         "manuscript from these numbers" % len(failed))


## The ledger

One entry per number in the paper. Partition and n travel with the value, so a figure can
never be quoted without the context that makes it meaningful.

In [ ]:
# ── Cell 5 · Build the manuscript number ledger ───────────────────────────────
# One entry per number that appears in the paper, each carrying its source
# notebook, its partition, its n, and where it is used. A number without a
# ledger entry does not go in the manuscript.

def entry(value, notebook, partition, n, appears_in, sd=None, note=None):
    e = {"value": value, "notebook": notebook, "partition": partition,
         "n": n, "appears_in": appears_in}
    if sd is not None: e["sd"] = sd
    if note: e["note"] = note
    return e

L = {}

# ── corpus and dataset ───────────────────────────────────────────────────────
if "corpus" in S:
    c = S["corpus"]["counts"]
    L["corpus_rag_documents"] = entry(c["rag_documents"], "NB01", "corpus", None,
                                      ["Table 2", "Section 3.7"])
    L["corpus_patient_cases"] = entry(c["patient_cases"], "NB01", "corpus", None,
                                      ["Section 3.7"])
    L["corpus_drugs"] = entry(
        S["corpus"]["provenance"]["rag_corpus"]["drugs_retrieved"], "NB01",
        "corpus", None, ["Section 3.7"])
if "attacks" in S:
    a = S["attacks"]["counts"]
    L["dataset_chunks"] = entry(a["chunks"], "NB02", "all", a["chunks"],
                                ["Table 2", "Figure 1", "Section 3.7"])
    L["dataset_benign"] = entry(a["benign"], "NB02", "all", a["chunks"],
                                ["Table 2", "Figure 1"])
    L["dataset_adversarial"] = entry(a["adversarial"], "NB02", "all", a["chunks"],
                                     ["Table 2", "Figure 1"])
    lp = S["attacks"]["length_parity"]
    L["length_parity_pct"] = entry(
        lp["median_pct_difference"], "NB02", "all", a["chunks"], ["Section 3.7"],
        note="R4-6: v1 had 2,648 vs 20 word medians")

# ── split ────────────────────────────────────────────────────────────────────
if "split" in S:
    for part in ["train", "val", "test"]:
        p = S["split"]["partitions"][part]
        L["split_%s_chunks" % part] = entry(p["chunks"], "NB03", part, p["chunks"],
                                            ["Table 2"])
        L["split_%s_adversarial" % part] = entry(p["adversarial"], "NB03", part,
                                                 p["chunks"], ["Table 2"])
    cp = S["split"]["confound_probes"]
    L["length_only_auc_v2"] = entry(cp["length_only_v2"]["auc"], "NB03", "test",
                                    cp["length_only_v2"]["n_test"], ["Section 4.1"],
                                    note="0.50 = no signal")
    L["length_only_auc_v1_counterfactual"] = entry(
        cp["length_only_v1_counterfactual"]["auc"], "NB03", "simulated", None,
        ["Section 4.1"], note="v1's distribution, same probe")

# ── Module 1 ─────────────────────────────────────────────────────────────────
if "module1" in S:
    r = S["module1"]["results"]
    for fmt, v in r.items():
        L["module1_%s_rate" % fmt] = entry(v["rate"], "NB04", "test probe",
                                           v["n"], ["Table 7"])
    av = S["module1"].get("adversarial_variants", {})
    if av.get("results"):
        L["module1_canonical_mean"] = entry(av["canonical_mean"], "NB04",
                                            "test probe", None, ["Table 7"])
        L["module1_adversarial_mean"] = entry(av["adversarial_mean"], "NB04",
                                              "test probe", None, ["Table 7"],
                                              note="Microsoft PyRIT suite")

# ── Module 2 ─────────────────────────────────────────────────────────────────
if "module2" in S:
    sm = S["module2"]["summary"]
    n_test = S["split"]["partitions"]["test"]["chunks"] if "split" in S else None
    for m in ["accuracy", "precision", "recall", "f1", "auc", "fpr"]:
        L["module2_%s" % m] = entry(sm[m]["mean"], "NB05", "test", n_test,
                                    ["Table 3", "Table 6"], sd=sm[m]["sd"])
    bd = S["module2"].get("breakdown", {}).get("vector", {})
    for vec, v in bd.items():
        L["module2_recall_%s" % vec] = entry(v["mean"], "NB05", "test", v["n"],
                                             ["Table 3", "Section 4.3"],
                                             sd=v["sd"])
    loc = S["module2"].get("localisation", {})
    if loc:
        L["attention_precision_at_k"] = entry(loc["precision_at_k_mean"], "NB05",
                                              "test", loc["n"], ["Figure 5"],
                                              sd=loc.get("precision_at_k_sd"))
        L["attention_random_baseline"] = entry(loc["random_baseline"], "NB05",
                                               "test", loc["n"], ["Figure 5"])

# ── Module 3 ─────────────────────────────────────────────────────────────────
if "module3" in S:
    t = S["module3"]["test"]
    n_test = S["split"]["partitions"]["test"]["chunks"] if "split" in S else None
    L["module3_recall"] = entry(t["combined"]["recall"], "NB06", "test", n_test,
                                ["Table 6"])
    L["module3_precision"] = entry(t["combined"]["precision"], "NB06", "test",
                                   n_test, ["Table 6"])
    sd = S["module3"]["similarity_distribution"]
    L["module3_similarity_auc_document"] = entry(
        sd["document_level"]["auc"], "NB06", "all", None, ["Section 4.4"],
        note="below chance: the similarity signal does not separate the classes")

# ── Module 4 and CEPS ────────────────────────────────────────────────────────
if "module4" in S:
    adv = S["module4"]["adversarial"]
    L["ceps_v1"] = entry(adv["ceps_v1"]["mean"], "NB07", "test", adv["n"],
                         ["Table 10"], sd=adv["ceps_v1"]["sd"])
    L["ceps_v2_preservation"] = entry(adv["preservation"]["mean"], "NB07", "test",
                                      adv["n"], ["Table 10"],
                                      sd=adv["preservation"]["sd"])
    L["ceps_v2_removal"] = entry(adv["removal"]["mean"], "NB07", "test", adv["n"],
                                 ["Table 10"], sd=adv["removal"]["sd"])
    L["ceps_v2_combined"] = entry(adv["combined"]["mean"], "NB07", "test", adv["n"],
                                  ["Table 10"], sd=adv["combined"]["sd"])
    st = S["module4"]["strategies"]
    noop = next((v for k, v in st.items() if "no-op" in k), None)
    if noop:
        L["ceps_v1_noop"] = entry(noop["ceps_v1"], "NB07", "test", adv["n"],
                                  ["Table 10"],
                                  note="R4-7: v1 scores a no-op sanitiser perfectly")
        L["ceps_v2_noop"] = entry(noop["combined"], "NB07", "test", adv["n"],
                                  ["Table 10"])
    ben = S["module4"]["benign"]
    L["sanitisation_benign_altered_pct"] = entry(
        ben["chunks_altered_pct"], "NB07", "test benign", ben["n"],
        ["Section 4.6"])
    pol = S["module4"].get("pipeline_policy", {})
    for k in ["sanitise_and_pass", "verify_or_block", "detect_and_block"]:
        if k in pol:
            L["policy_%s_residual_pct" % k] = entry(
                pol[k].get("residual_risk_pct", pol[k].get("payload_survives_pct")),
                "NB07", "test", adv["n"], ["Table 11"])

# ── baselines ────────────────────────────────────────────────────────────────
if "baselines" in S:
    n_test = S["baselines"]["protocol"]["n_test"]
    for name, r in S["baselines"]["baselines"].items():
        for m in ["accuracy", "recall", "fpr", "f1"]:
            L["baseline_%s_%s" % (name, m)] = entry(r[m], "NB08", "test", n_test,
                                                    ["Table 4"])
    ext = S["baselines"].get("external_transfer", {})
    inj = ext.get("injected_payloads", {}).get("results", {})
    for corpus, r in inj.items():
        L["transfer_%s_recall" % corpus] = entry(
            r["recall"]["mean"], "NB08", "test hosts", r.get("n_hosts"),
            ["Table 5"], sd=r["recall"]["sd"],
            note="external payloads injected into clinical documents")

# ── end to end ───────────────────────────────────────────────────────────────
if "end_to_end" in S:
    e = S["end_to_end"]
    n = e["n_samples"]
    for c in ["clean", "poisoned", "sanitised"]:
        L["asr_%s" % c] = entry(e["asr"][c], "NB09", "test", n,
                                ["Table 9", "Abstract"])
    L["asr_relative_reduction"] = entry(e["asr"]["relative_reduction"], "NB09",
                                        "test", n, ["Table 9", "Abstract"],
                                        note="the evidence for 'Neutralizing'")
    L["asr_mcnemar_p"] = entry(e["mcnemar"]["p"], "NB09", "test", n, ["Table 9"])
    L["target_llm"] = entry(e["target_model"], "NB09", None, None, ["Section 3.9"])

# ── MPIB ─────────────────────────────────────────────────────────────────────
if "mpib" in S:
    m = S["mpib"]
    L["mpib_detection_rate"] = entry(m["detection"]["rate"], "NB10", "V2",
                                     m["benchmark"]["n_evaluated"], ["Table 5"],
                                     note="R1-d, R4-2: v1 reported 99-100%, which "
                                          "was the attack success rate")
    for c in ["clean", "poisoned", "sanitised"]:
        L["mpib_asr2_%s" % c] = entry(m["mpib_metrics"][c]["ASR2_op"], "NB10", "V2",
                                      m["benchmark"]["n_evaluated"], ["Table 5"])
        L["mpib_cher3_%s" % c] = entry(m["mpib_metrics"][c]["CHER3_op"], "NB10",
                                       "V2", m["benchmark"]["n_evaluated"],
                                       ["Table 5"])

# ── sweeps ───────────────────────────────────────────────────────────────────
if "sweeps" in S:
    for sweep, rows in S["sweeps"]["results"].items():
        for lab, v in rows.items():
            key = "sweep_%s_%s" % (sweep, re.sub(r"[^a-z0-9]+", "_", lab.lower()))
            L[key] = entry(v["accuracy"], "NB11", "test", None, ["Table 12"],
                           sd=v["accuracy_sd"])

print("ledger entries: %d" % len(L))
print("")
by_nb = Counter(v["notebook"] for v in L.values())
for nb in sorted(by_nb):
    print("  %-6s %3d numbers" % (nb, by_nb[nb]))
print("")
no_n = [k for k, v in L.items() if v["n"] is None and v["partition"] not in (None,)]
if no_n:
    print("  %d entries carry no n -- acceptable only where the quantity has none"
          % len(no_n))


## Reviewer coverage

All 31 points, each mapped to the notebook that addresses it and the numbers that evidence
it. Points marked `NO DATA` have no ledger entry yet and cannot be claimed as resolved.

In [ ]:
# ── Cell 6 · Reviewer coverage map ────────────────────────────────────────────
# Each reviewer point mapped to the notebook that addresses it and the ledger
# entries that evidence it. This is the skeleton of the point-by-point response.

COVERAGE = {
    "R1-a": ("confusion matrix vs Table 6 inconsistent", "NB05",
             ["module2_accuracy", "module2_fpr", "module2_recall"],
             "every metric recomputed from the pooled matrix and asserted equal"),
    "R1-b": ("Table 2 vs Figure 3 vs test-set size", "NB01/NB02/NB03",
             ["dataset_chunks", "split_train_chunks", "split_val_chunks",
              "split_test_chunks"],
             "partition sizes derived from one manifest and cross-checked"),
    "R1-c": ("CEPS values and sample counts inconsistent", "NB07",
             ["ceps_v1", "ceps_v2_combined"],
             "every CEPS figure carries its partition and n"),
    "R1-d": ("MPIB ASR reported as detection", "NB10",
             ["mpib_detection_rate", "mpib_asr2_poisoned"],
             "detection and ASR reported as separate quantities"),
    "R1-e": ("encoded variants split across partitions", "NB02/NB03",
             ["split_train_chunks"],
             "group_id assigned at corpus build, inherited by every variant; "
             "asserted at split time"),
    "R1-f": ("'neutralizing' unevidenced end to end", "NB09",
             ["asr_poisoned", "asr_sanitised", "asr_relative_reduction",
              "asr_mcnemar_p"],
             "attack success measured on a target LLM before and after"),
    "R2-1": ("payloads concatenated, testing direct not indirect injection", "NB02",
             ["dataset_adversarial"],
             "attacks injected into retrieved documents"),
    "R2-2": ("Deepset payloads are generic, not clinical", "NB02",
             ["dataset_adversarial"],
             "four clinically specific categories, drug-aware"),
    "R2-3": ("PubMed abstracts are not clinical data", "NB01",
             ["corpus_rag_documents", "corpus_drugs"],
             "openFDA structured drug labelling"),
    "R2-4": ("Figure 1 redundant with Table 1", "manuscript", [],
             "figure removed, downstream figures renumbered"),
    "R3-seeds": ("single training run", "NB05",
                 ["module2_accuracy", "module2_recall"],
                 "five seeds, mean +/- SD and 95% CI throughout"),
    "R3-Q1": ("how does it improve on existing methods", "NB08",
              ["baseline_tfidf_lr_accuracy", "module2_accuracy"], ""),
    "R3-Q2": ("five obfuscation formats and their validation", "NB04",
              ["module1_base64_rate", "module1_leet_rate"],
              "per-format rates with Wilson intervals, n stated"),
    "R3-Q3": ("attention reliability vs alternatives", "NB05",
              ["attention_precision_at_k", "attention_random_baseline"],
              "precision@k against a random baseline"),
    "R3-Q4": ("computational overhead and real-time feasibility", "NB05", [], ""),
    "R3-Q5": ("CEPS validated against clinicians", "not run", [],
              "stated in Limitations as un-validated against expert judgement"),
    "R3-Q6": ("reproducibility of datasets and protocols", "repository", [],
              "corpus, splits, modules and results deposited"),
    "R3-Q7": ("justification for 70/15/15", "NB11", [],
              "swept against 80/10/10 and 60/20/20; within seed spread"),
    "R3-Q8": ("checkpoint on accuracy, early stop on loss", "NB11", [],
              "one criterion now governs both; the difference is within spread"),
    "R3-Q9": ("rationale for patience=3", "NB11", [],
              "swept 2, 3 and 5; within seed spread"),
    "R3-Q10": ("quantitative localisation metrics and failure cases", "NB05",
               ["attention_precision_at_k"],
               "precision@k, IoU, random baseline, failure cases printed"),
    "R3-MoE": ("head-to-head against Mixture of Encodings", "NB08",
               ["baseline_mixture_of_encodings_recall"],
               "benchmarked; prevention-only, emits no detection signal"),
    "R3-decode": ("ablation: does decoding before detection help", "NB11", [],
                  "no measurable effect on Module 2; value is in NB04 robustness"),
    "R4-1": ("text classification, not end-to-end", "NB09",
             ["asr_poisoned", "asr_sanitised"], ""),
    "R4-2": ("ASR and CHER treated as detector-miss rates", "NB10",
             ["mpib_detection_rate", "mpib_asr2_poisoned", "mpib_cher3_poisoned"], ""),
    "R4-3": ("Figure 6 implies 86.98% vs Table 6's 92.23%", "NB05/NB12",
             ["module2_accuracy", "module2_fpr"],
             "reconciliation gate in Cell 4"),
    "R4-4": ("baselines conflict with repository JSON", "NB08",
             ["baseline_tfidf_lr_accuracy"],
             "each row asserted consistent with its own matrix"),
    "R4-5": ("Figure 3 totals 6,453 vs Table 2's 6,627", "NB12",
             ["dataset_chunks"], "single source, cross-checked"),
    "R4-6": ("length confounding", "NB02/NB03",
             ["length_parity_pct", "length_only_auc_v2",
              "length_only_auc_v1_counterfactual"],
             "removed by construction and measured"),
    "R4-7": ("CEPS scores no-op sanitisation perfectly", "NB07",
             ["ceps_v1_noop", "ceps_v2_noop", "ceps_v2_removal"],
             "CEPS v2 pairs preservation with removal"),
    "R4-8": ("repository omits Module 3, Module 5, scripts, checkpoints", "repo",
             [], "modules exported to src/ as each notebook runs"),
    "R4-9": ("MPIB restricted payload registry redistributed", "repo", [],
             "removed from the working tree and from git history on both "
             "branches; .gitignore added"),
    "R4-10": ("Figure 7 caption contradicts the figure", "NB05",
              ["attention_precision_at_k"],
              "figure generated from a confirmed true positive"),
}

resolved = unresolved = 0
print("  point     status      evidence")
print("  " + "-"*78)
for pid, (desc, where, keys, note) in sorted(COVERAGE.items()):
    have = [k for k in keys if k in L]
    if where in ("not run",):
        status = "stated"
    elif keys and not have:
        status = "NO DATA"
    else:
        status = "resolved"
    resolved += status == "resolved"
    unresolved += status == "NO DATA"
    print("  %-9s %-11s %s%s"
          % (pid, status, where,
             ("  [%d numbers]" % len(have)) if have else ""))

print("")
print("  %d of %d points backed by ledger numbers; %d awaiting data"
      % (resolved, len(COVERAGE), unresolved))


## Output

`all_paper_numbers.json` is the manuscript's source of truth. `all_paper_numbers.csv` is the
flat version, for checking the written paper line by line against the measured values.

In [ ]:
# ── Cell 7 · Write the manuscript source of truth ─────────────────────────────
all_numbers = {
    "notebook": "12_aggregation",
    "seed": SEED,
    "purpose": ("Single source for every number in the manuscript. v1's tables "
                "were transcribed by hand from notebook output and drifted from "
                "the figures they described; Reviewers 1 and 4 found four such "
                "inconsistencies. Nothing here is retyped."),
    "sources": {k: CONFIG["SOURCES"][k] for k in S},
    "missing_sources": missing,
    "reconciliation": {"checks": checks,
                       "passed": sum(c["passed"] for c in checks),
                       "total": len(checks)},
    "ledger": L,
    "reviewer_coverage": {k: {"description": v[0], "addressed_in": v[1],
                              "ledger_keys": [x for x in v[2] if x in L],
                              "note": v[3]}
                          for k, v in COVERAGE.items()},
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
}

out = DIRS["results"] / "all_paper_numbers.json"
with open(out, "w") as f:
    json.dump(all_numbers, f, indent=2)
print("written: %s (%.0f KB, %d ledger entries)"
      % (out.name, out.stat().st_size/1024, len(L)))

# a flat CSV, for checking the manuscript against line by line
import csv as _csv
csv_path = DIRS["results"] / "all_paper_numbers.csv"
with open(csv_path, "w", newline="") as f:
    w = _csv.writer(f)
    w.writerow(["key", "value", "sd", "notebook", "partition", "n",
                "appears_in", "note"])
    for k, v in sorted(L.items()):
        w.writerow([k, v["value"], v.get("sd", ""), v["notebook"],
                    v["partition"] or "", v["n"] if v["n"] is not None else "",
                    "; ".join(v["appears_in"]), v.get("note", "")])
print("written: %s" % csv_path.name)

if IN_COLAB and DRIVE_ROOT:
    dst = DRIVE_ROOT / "data" / "results"; dst.mkdir(parents=True, exist_ok=True)
    for f_ in [out, csv_path]:
        shutil.copy2(f_, dst / f_.name)
    print("mirrored to Drive")

if IN_COLAB and CONFIG["PUSH_TO_GITHUB"] and REPO_DIR and REPO_DIR.exists():
    subprocess.run(["git","-C",str(ROOT),"add","-f",
                    "data/results/all_paper_numbers.json",
                    "data/results/all_paper_numbers.csv"], check=False)
    st = subprocess.run(["git","-C",str(ROOT),"status","--porcelain"],
                        capture_output=True, text=True)
    if st.stdout.strip():
        subprocess.run(["git","-C",str(ROOT),"commit","-q","-m",
                        "NB12: %d numbers reconciled, %d/%d cross-checks passed"
                        % (len(L), sum(c["passed"] for c in checks), len(checks))],
                       check=False)
        pr = subprocess.run(["git","-C",str(ROOT),"push","-q","origin",
                             CONFIG["GITHUB_BRANCH"]], capture_output=True, text=True)
        print("push:", "ok" if pr.returncode == 0 else pr.stderr[:200])


## Summary

**Paste the output below into the chat.**

Two things decide whether the pipeline is complete: every cross-check passing, and no
reviewer point left awaiting data.

In [ ]:
# ── Cell 8 · NB12 SUMMARY — paste this output ─────────────────────────────────
print("=" * 74)
print("NB12 — AGGREGATION AND RECONCILIATION")
print("=" * 74)
print("sources loaded  : %d of %d" % (len(S), len(CONFIG["SOURCES"])))
if missing:
    print("sources MISSING : %s" % missing)
print("ledger entries  : %d" % len(L))
print("cross-checks    : %d/%d passed" % (sum(c["passed"] for c in checks), len(checks)))
print("")
print("HEADLINE NUMBERS")
def show(key, label, pct=False, dp=4):
    if key not in L: 
        print("  %-38s (not available)" % label); return
    v = L[key]
    val = v["value"]
    s = ("%.1f%%" % (100*val)) if pct else (("%.*f" % (dp, val)) if isinstance(val, (int,float)) else str(val))
    sd = ("  +/- %.4f" % v["sd"]) if v.get("sd") is not None else ""
    n = ("  n=%s" % v["n"]) if v["n"] else ""
    print("  %-38s %s%s%s   [%s]" % (label, s, sd, n, v["notebook"]))

show("dataset_chunks", "dataset chunks")
show("dataset_adversarial", "adversarial chunks")
show("length_parity_pct", "length parity difference (%)")
show("length_only_auc_v2", "length-only AUC, v2 corpus")
show("length_only_auc_v1_counterfactual", "length-only AUC, v1 design")
print("")
show("module2_accuracy", "Module 2 accuracy")
show("module2_recall", "Module 2 recall")
show("module2_fpr", "Module 2 FPR")
show("module2_recall_override", "  recall, override attacks")
show("module2_recall_misinformation", "  recall, misinformation attacks")
print("")
show("module1_canonical_mean", "Module 1, canonical formats", pct=True)
show("module1_adversarial_mean", "Module 1, PyRIT adversarial", pct=True)
show("module3_recall", "Module 3 recall")
print("")
show("ceps_v1", "CEPS v1")
show("ceps_v1_noop", "CEPS v1 on a no-op sanitiser")
show("ceps_v2_combined", "CEPS v2")
show("ceps_v2_noop", "CEPS v2 on a no-op sanitiser")
print("")
show("asr_clean", "ASR, clean documents")
show("asr_poisoned", "ASR, poisoned documents")
show("asr_sanitised", "ASR, after sanitisation")
show("asr_relative_reduction", "ASR relative reduction", pct=True)
show("asr_mcnemar_p", "McNemar p", dp=8)
print("")
show("mpib_detection_rate", "MPIB V2 detection rate")
show("mpib_asr2_poisoned", "MPIB ASR2, poisoned")
show("mpib_asr2_clean", "MPIB ASR2, clean")
print("")
print("REVIEWER COVERAGE")
res = sum(1 for k, v in COVERAGE.items()
          if v[1] != "not run" and (not v[2] or any(x in L for x in v[2])))
print("  %d of %d points backed by ledger numbers" % (res, len(COVERAGE)))
gaps = [k for k, v in COVERAGE.items() if v[2] and not any(x in L for x in v[2])]
if gaps:
    print("  awaiting data: %s" % ", ".join(gaps))
print("=" * 74)
print("NB12 COMPLETE — the manuscript is written from all_paper_numbers.json")
print("=" * 74)
